In [1]:
# 1. imports & setup
from google import genai
from google.genai import types
import json
import re
import os
from dotenv import load_dotenv
from warnings import filterwarnings

filterwarnings("ignore")
load_dotenv()

True

In [3]:
# 2. create client
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL_NAME = os.getenv("GEMINI_MODEL_NAME")

client = genai.Client(api_key = GEMINI_API_KEY)

In [6]:
# 3. create a prompt template
PROMPT_TEMPLATE = """
You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {{
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }}
```
User Message:
{}
"""

In [24]:
# 4. handle input
# user_input = "Your package delivery failed. Please update your address at [www.bestwebsite.com] to avoid return to sender."
# user_input = "CONGRATULATIONS!! You have won a lottery of 10 million dollars!! Give me your OTP now and take the cash."
# user_input = "Your gmail password is expiring, clicking on the link below to reset the password"
# user_input = "Your ITR Refund is due Enter the account details today to claim"
# user_input = "We found some restricted items in your package. Send your address and CREDIT CARD to resolve this now!"
user_input = "You have an unclaimed policy for $10000, fill the details to claim it, otherwise it will expire in the next 2 hours."

full_prompt = PROMPT_TEMPLATE.format(user_input)
print(full_prompt)


You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }
```
User Message:
Your gmail password is expiring, clicking on the link below to reset the password



In [25]:
# 5. generate llm response
response = client.models.generate_content(
    model = GEMINI_MODEL_NAME,
    contents = full_prompt
)

print(response.text)

1. **Thought**: The message attempts to create urgency by stating the Gmail password is expiring. It then provides a direct call to action with a link to "reset the password." This is a common tactic used in phishing scams to trick users into clicking malicious links and revealing their login credentials. The message lacks any personalization or official sender identification, which are also typical of scam attempts.

2. **Action**: This message exhibits several common characteristics of phishing scams. The urgency, the generic nature, and the prompt to click a link for password reset are all strong indicators of a potential scam.

3. **Final Answer**:
```json
{
  "label": "Scam",
  "reasoning": "The message creates a false sense of urgency by stating the Gmail password is expiring. It then provides a generic link to reset the password, which is a common phishing tactic. Legitimate services typically do not send password expiry warnings with direct links for reset. The message lacks pe

In [12]:
# 6. parse output
def extract_json(text):
    """
    Finds the first '{' and the last '}' in a string 
    and parses it as JSON.
    """
    try:
        # Regex to find the JSON block inside the text
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"JSON Parsing Error: {e}")
    
    # Fallback if parsing fails
    return {
        "label": "Uncertain", 
        "reasoning": "Could not parse AI response",
        "intent": "Unknown",
        "risk_factors": ["Parsing Error"]
    }

parsed_output = extract_json(response.text)

In [17]:
# 7. print output
parsed_output

{'label': 'Scam',
 'reasoning': "The message creates artificial urgency by stating the policy will expire in 2 hours, which is a common scam tactic. It also offers an unsolicited, large sum of money ($10000) for an 'unclaimed policy' without providing any context or legitimate origin. The instruction to 'fill the details' strongly suggests a phishing attempt to collect personal or financial information.",
 'intent': 'To trick the recipient into revealing personal and financial information by claiming a fake, unclaimed policy before it supposedly expires.',
 'risk_factors': ['Urgency (2-hour expiry)',
  'Unsolicited large sum of money offer',
  'Vague origin of the policy',
  "Request for personal/financial details ('fill the details')"]}

In [13]:
type(parsed_output)

dict

In [19]:
print(parsed_output['label'])
print()
print(parsed_output['reasoning'])

Scam

The message creates artificial urgency by stating the policy will expire in 2 hours, which is a common scam tactic. It also offers an unsolicited, large sum of money ($10000) for an 'unclaimed policy' without providing any context or legitimate origin. The instruction to 'fill the details' strongly suggests a phishing attempt to collect personal or financial information.
